## Why Camera Calibration?
### To Remove Disortion on our camera
- 1. Barrel Disortion
- 2. Pincushion Disortion

### Zhang's Method - Linear Solution for pinHole Camera

## 1. Import Statements

In [1]:
import cv2
import numpy as np
import os

## 2. Find Chessboard Corners - objPoints and imgPoints

In [2]:
# Height Width
chessboardSize = (24, 17)
frameSize = (1440, 1080)

# termination criteria
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

# Prepare Object Points Eg (0,0,0) (1,0,0)
objp = np.zeros((chessboardSize[0]*chessboardSize[1], 3), np.float32)
objp[:,:2] = np.mgrid[0:chessboardSize[0], 0:chessboardSize[1]].T.reshape(-1,2)

# If you don't multiply objp by the size of squares, the calibration process will still work, 
# but the resulting intrinsic parameters (e.g., focal length) will be in pixel units, 
# and any measurements you make using the calibrated camera will also be in pixel units 
# instead of real-world units.

# Arrays to Store Object Points & Image Points from all images
objPoints = [] # 3d point in real world space
imgPoints = [] # 2d points in image plane

all_images = 'Images/Calibration/input'
images = os.listdir(all_images)

for image in images:
    print(f'{all_images}/{image}')
    img = cv2.imread(f'{all_images}/{image}')
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Find Chessboard Corners
    ret, corners = cv2.findChessboardCorners(gray, chessboardSize, None)
    
    if ret == True:
        
        objPoints.append(objp)
        corners2 = cv2.cornerSubPix(gray, corners, (11,11), (-1,-1), criteria)
        imgPoints.append(corners)
        
        # Draw and display the corners
        cv2.drawChessboardCorners(img, chessboardSize, corners2, ret)
        cv2.imshow('Image', img)
        cv2.waitKey(0)
        
cv2.destroyAllWindows()

Images/Calibration/input/cali1.png
Images/Calibration/input/cali2.png
Images/Calibration/input/cali3.png
Images/Calibration/input/cali4.png
Images/Calibration/input/cali5.png


## 3. Calibration

In [3]:
ret, cameraMatrix, dist, rvecs, tvecs = cv2.calibrateCamera(objPoints, imgPoints, frameSize, None, None)

print('Camera Calibrated: ', ret)
print('\nCamera Matrix:\n', cameraMatrix)
print('\nDistortion Parameters:\n', dist)
print('\nRotation Vectors:\n', rvecs)
print('\nTransalation Vectors:\n', tvecs)

Camera Calibrated:  1.3211328758489709

Camera Matrix:
 [[1.14700909e+03 0.00000000e+00 7.47677098e+02]
 [0.00000000e+00 1.14732821e+03 5.87450375e+02]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]

Distortion Parameters:
 [[-0.23502292  0.13565442 -0.00122041 -0.00225529 -0.08375011]]

Rotation Vectors:
 (array([[-0.0224564 ],
       [-0.00232134],
       [-1.58867208]]), array([[ 0.31029743],
       [ 0.0012552 ],
       [-0.01844264]]), array([[-0.01965001],
       [ 0.00516733],
       [-0.0013305 ]]), array([[-0.00841527],
       [ 0.35573643],
       [-0.05731921]]), array([[-0.01735929],
       [ 0.00542362],
       [ 0.01649278]]))

Transalation Vectors:
 (array([[-20.66107304],
       [ 22.11800738],
       [ 68.53228643]]), array([[-26.9082994 ],
       [ 10.80065633],
       [ 62.9402317 ]]), array([[-45.28147123],
       [-16.01898771],
       [ 69.20888162]]), array([[-22.43978531],
       [-17.2136594 ],
       [ 68.62469959]]), array([[ 0.18088128],
       [10.6638825

## 4. Undistortion

In [4]:
img = cv2.imread('Images/Calibration/input/cali5.png')
h, w = img.shape[:2]
newCameraMatrix, roi = cv2.getOptimalNewCameraMatrix(cameraMatrix, dist, (w,h), 1, (w,h))

# Undistort
dst = cv2.undistort(img, cameraMatrix, dist, None, newCameraMatrix)

# Crop the Image
x, y, w, h = roi
dst = dst[y:y+h, x:x+w]
cv2.imwrite('Images/Calibration/CaliResult1.png', dst)

# Undistort with Remapping
mapx, mapy = cv2.initUndistortRectifyMap(cameraMatrix, dist, None, newCameraMatrix, (w,h), 5)
dst = cv2.remap(img, mapx, mapy, cv2.INTER_LINEAR)

# Crop the Image
x, y, w, h = roi
dst = dst[y:y+h, x:x+w]
cv2.imwrite('Images/Calibration/CaliResult2.png', dst)

## 5. Reprojection Error

In [5]:
mean_error = 0

for i in range(len(objPoints)):
    imgPoints2, _ = cv2.projectPoints(objPoints[i], rvecs[i], tvecs[i], cameraMatrix, dist)
    error = cv2.norm(imgPoints[i], imgPoints2, cv2.NORM_L2)/len(imgPoints2)
    mean_error += error
    
print('Total Error: {}'.format(mean_error/len(objPoints)))

Total Error: 0.04990803784730058
